# Module 3: Baseline Forecasting Models

This notebook builds baseline forecasts and evaluates them using business-friendly metrics.

## What you'll do
- Load feature-ready dataset from Module 2
- Create a time-based train/test split
- Run baseline models (naive, moving average, SES)
- Evaluate with MAE, RMSE, WAPE, sMAPE
- Produce an error comparison table


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../')

from src.models.baseline import (
    forecast_by_sku,
    naive_forecast,
    moving_average_forecast,
    simple_exponential_smoothing_forecast,
)
from src.evaluation.metrics import mae, rmse, wape, smape

print('Imports OK')


Imports OK


## Load feature-ready data (from Module 2)

We expect the output file:
- `data/processed/featured_sales_data.csv`


In [2]:
data_path_candidates = [
    Path('../data/processed/featured_sales_data.csv'),
    Path('data/processed/featured_sales_data.csv'),
]

data_path = next((p for p in data_path_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find featured_sales_data.csv. Run Module 2 to generate it: notebooks/02_data_cleaning.ipynb"
    )

df = pd.read_csv(data_path, parse_dates=['date'])
print(f"Loaded: {data_path}")
print(df.shape)
df.head()


Loaded: ..\data\processed\featured_sales_data.csv
(361500, 65)


,sku_id,date,category,subcategory,price,units_sold,revenue,promotion_flag,stock_available,year,...,price_rolling_mean_7,price_relative_to_avg_7,price_rolling_mean_30,price_relative_to_avg_30,price_rolling_std_7,month_day_interaction,holiday_weekend,promo_weekend,is_peak_period,quarter_day_interaction
0,SKU001,2023-12-24,Electronics,Accessories,168.81,28,4726.68,0,108,2023,...,157.494286,1.071848,157.220000,1.073718,14.218150,72,1,0,1,24
1,SKU001,2023-12-25,Electronics,Accessories,167.59,20,3351.89,0,6,2023,...,158.820000,1.055220,158.372222,1.058203,14.730299,0,0,0,0,0
2,SKU001,2023-12-26,Electronics,Accessories,165.31,22,3636.74,0,198,2023,...,164.227143,1.006594,159.066000,1.039254,5.097391,12,0,0,0,4
3,SKU001,2023-12-27,Electronics,Accessories,164.80,46,7580.63,0,52,2023,...,165.731429,0.994380,159.587273,1.032664,2.621821,24,0,0,0,8
4,SKU001,2023-12-28,Electronics,Accessories,131.47,33,4338.57,1,123,2023,...,161.214286,0.815498,157.244167,0.836088,13.324664,36,0,0,0,12


## Define train/test split

We’ll do a simple cutoff split:
- Train: all dates <= cutoff
- Test: next `horizon` days

This mirrors real forecasting usage (predict the future).


In [3]:
HORIZON = 14  # days

# Choose a cutoff so we have at least HORIZON days after it
max_date = df['date'].max()
cutoff = max_date - pd.Timedelta(days=HORIZON)

print(f"Max date: {max_date.date()}")
print(f"Cutoff date: {cutoff.date()} (forecast next {HORIZON} days)")

train = df[df['date'] <= cutoff].copy()
test = df[(df['date'] > cutoff) & (df['date'] <= cutoff + pd.Timedelta(days=HORIZON))].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train date range:", train['date'].min().date(), "->", train['date'].max().date())
print("Test date range:", test['date'].min().date(), "->", test['date'].max().date())

assert test['date'].nunique() == HORIZON, "Test window should be exactly HORIZON unique dates"


Max date: 2025-12-15
Cutoff date: 2025-12-01 (forecast next 14 days)
Train shape: (354500, 65)
Test shape: (7000, 65)
Train date range: 2023-12-24 -> 2025-12-01
Test date range: 2025-12-02 -> 2025-12-15


## Run baseline models

We’ll forecast **per SKU** (local models):
- Naive (last value)
- Moving Average (window=7)
- SES (alpha=0.3)

Then we’ll join predictions with actuals in the test window.


In [4]:
SKU_COL = 'sku_id'
DATE_COL = 'date'
TARGET_COL = 'units_sold'

# Build forecasts for each method
pred_naive = forecast_by_sku(train, sku_col=SKU_COL, date_col=DATE_COL, target_col=TARGET_COL,
                             horizon=HORIZON, method='naive', cutoff_date=cutoff)

pred_ma7 = forecast_by_sku(train, sku_col=SKU_COL, date_col=DATE_COL, target_col=TARGET_COL,
                           horizon=HORIZON, method='moving_average', ma_window=7, cutoff_date=cutoff)

pred_ses = forecast_by_sku(train, sku_col=SKU_COL, date_col=DATE_COL, target_col=TARGET_COL,
                           horizon=HORIZON, method='ses', ses_alpha=0.3, cutoff_date=cutoff)

preds = pd.concat([pred_naive, pred_ma7, pred_ses], ignore_index=True)

# Join with actuals
actuals = test[[SKU_COL, DATE_COL, TARGET_COL]].rename(columns={TARGET_COL: 'y_true'})
scored = preds.merge(actuals, on=[SKU_COL, DATE_COL], how='inner')

print('Pred rows:', len(preds))
print('Scored rows:', len(scored))
scored.head()


Pred rows: 21000
Scored rows: 21000


,sku_id,date,y_pred,method,y_true
0,SKU001,2025-12-02,15.0,naive,28
1,SKU001,2025-12-03,15.0,naive,35
2,SKU001,2025-12-04,15.0,naive,0
3,SKU001,2025-12-05,15.0,naive,21
4,SKU001,2025-12-06,15.0,naive,42


## Evaluate baselines

We’ll compute overall metrics per model and build a comparison table.

Notes:
- **WAPE** is often the most business-friendly.
- **sMAPE** is useful when there are many low/zero values.


In [5]:
def compute_metrics(df_part: pd.DataFrame) -> dict:
    return {
        'MAE': mae(df_part['y_true'], df_part['y_pred']),
        'RMSE': rmse(df_part['y_true'], df_part['y_pred']),
        'WAPE': wape(df_part['y_true'], df_part['y_pred']),
        'sMAPE': smape(df_part['y_true'], df_part['y_pred']),
    }

results = []
for method, g in scored.groupby('method'):
    m = compute_metrics(g)
    m['method'] = method
    results.append(m)

summary = pd.DataFrame(results).set_index('method').sort_values('WAPE')
summary


,MAE,RMSE,WAPE,sMAPE
method,,,,
moving_average,20.711163,28.624601,0.449279,0.575401
ses,20.782125,28.853977,0.450818,0.574644
naive,26.483286,36.952833,0.574491,0.783392


## Optional: Per-SKU error distribution

This helps you see whether a model is good overall but bad for many SKUs (or vice versa).


In [6]:
# Compute per-SKU WAPE per method
per_sku = (
    scored.groupby(['method', SKU_COL])
    .apply(lambda x: wape(x['y_true'], x['y_pred']))
    .reset_index(name='WAPE')
)

per_sku.groupby('method')['WAPE'].describe()


,count,mean,std,min,25%,50%,75%,max
method,,,,,,,,
moving_average,500.0,0.472879,0.152398,0.148025,0.364970,0.457374,0.555593,1.243632
naive,500.0,0.607591,0.264971,0.160334,0.400955,0.522264,0.797513,1.503311
ses,500.0,0.477020,0.156805,0.184675,0.357374,0.455462,0.562426,1.278261


In [7]:
# Save baseline summary
out_dir = Path('../outputs/reports')
out_dir.mkdir(parents=True, exist_ok=True)

summary_out = out_dir / 'module3_baseline_summary.csv'
summary.reset_index().to_csv(summary_out, index=False)
print('Saved:', summary_out)

# Save per-sku distribution
per_sku_out = out_dir / 'module3_baseline_per_sku_wape.csv'
per_sku.to_csv(per_sku_out, index=False)
print('Saved:', per_sku_out)


Saved: ..\outputs\reports\module3_baseline_summary.csv
Saved: ..\outputs\reports\module3_baseline_per_sku_wape.csv
